In [ ]:
import os
import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import math

# KONFIGURASI
MODEL_PATH = "D:/skripsi/wastecategorized13.tflite"
TEST_DIR = "D:/skripsi/test"
OUTPUT_DIR = "D:/skripsi/output3"
IMG_SIZE = 224

os.makedirs(OUTPUT_DIR, exist_ok=True)


CLASS_NAMES =  sorted(os.listdir(TEST_DIR))

# LOAD MODEL
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()


# PREPROCESS
def preprocess_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)
    return img

# KUMPULKAN SEMUA GAMBAR

all_images = []

for class_name in CLASS_NAMES:
    class_folder = os.path.join(TEST_DIR, class_name)
    for file in os.listdir(class_folder):
        img_path = os.path.join(class_folder, file)
        all_images.append((img_path, class_name, file))

total_images = len(all_images)
print("Total gambar:", total_images)


# GRID DINAMIS
cols = 1
rows = math.ceil(total_images / cols)

plt.figure(figsize=(10, rows * 4))


# LOOP SEMUA GAMBAR
for index, (img_path, actual_class, file_name) in enumerate(all_images, start=1):

    # Load original image (untuk ditampilkan & disimpan)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    input_data = preprocess_image(img_path)

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()

    output_data = interpreter.get_tensor(output_details[0]['index'])
    pred_index = np.argmax(output_data)
    pred_class = CLASS_NAMES[pred_index]

   # Warna teks
    color_cv = (0, 255, 0) if pred_class == actual_class else (0, 0, 255)
    color_plt = "green" if pred_class == actual_class else "red"


    # TAMBAH HEADER
    header_height = 80
    header = np.zeros((header_height, img.shape[1], 3), dtype=np.uint8)

    img_with_header = np.vstack((header, img))

    cv2.putText(img_with_header,
                f"Actual   : {actual_class}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8, color_cv, 2)

    cv2.putText(img_with_header,
                f"Predicted: {pred_class}",
                (10, 65),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8, color_cv, 2)

    # Simpan hasil
    output_name = f"{os.path.splitext(file_name)[0]}_{actual_class}_{pred_class}.png"
    output_path = os.path.join(OUTPUT_DIR, output_name)

    cv2.imwrite(output_path, img_with_header)

    # =============================
    # TAMPILKAN DI MATPLOTLIB
    # =============================
    plt.subplot(rows, cols, index)
    plt.imshow(img_rgb)
    plt.title(f"A: {actual_class}\nP: {pred_class}", color=color_plt)
    plt.axis("off")

plt.tight_layout()
plt.show()
